In [ ]:
import os
import pickle
import lmdb
import pandas as pd
import numpy as np
from rdkit import Chem
from tqdm import tqdm
from rdkit.Chem import AllChem
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')  
import warnings
warnings.filterwarnings(action='ignore')
from multiprocessing import Pool
import json
import random
import shutil
import copy
import torch

from sklearn.model_selection import KFold, StratifiedKFold

In [ ]:
# random seed
seed = 42
random.seed(seed)
np.random.seed(seed)


In [ ]:
# open train/validtest.json to get lnp_id

json_folder = 'processed_data_dirs/demo_in_house_lnp_data_overall_new_full_with_pbae_NPratios_updated_09222023_npratios_09252023gen_fig4bii/fold_V0/in_house_lnp/'

nonPBAE_lnp_ids_dict = {}
PBAE_lnp_ids_dict = {}
all_lnp_ids_dict = {}

# iterate over all files in the json_folder
for f in os.listdir(json_folder):
    if f.endswith('train.json') or f.endswith('valid.json') or f.endswith('test.json'):
        print("f: ", f)
        nonPBAE_lnp_ids_dict[f] = []
        PBAE_lnp_ids_dict[f] = []
        all_lnp_ids_dict[f] = []

        with open(os.path.join(json_folder, f), 'r') as fp:
            data = json.load(fp)

            for lnp in data:
                lnp_id = lnp['lnp_id']
                components = lnp['components']
                nonPBAE = True
                for c in components:
                    if 'PBAE' in c['component_type']:
                        nonPBAE = False
                        break
                
                if nonPBAE:
                    nonPBAE_lnp_ids_dict[f].append(lnp_id)
                else:
                    PBAE_lnp_ids_dict[f].append(lnp_id)

                all_lnp_ids_dict[f].append(lnp_id)



In [ ]:
nonPBAE_lnp_ids_dict['train.json']

In [ ]:
PBAE_lnp_ids_dict['train.json']

In [ ]:
PBAE_lnp_train_size = len(PBAE_lnp_ids_dict['train.json'])

In [ ]:
PBAE_subset_pcts = [5, 25, 50, 75]

for p in PBAE_subset_pcts:
    print("p: ", p)
    PBAE_subset_size = int(PBAE_lnp_train_size * p / 100)
    print("PBAE_subset_size: ", PBAE_subset_size)

    PBAE_subset = random.sample(PBAE_lnp_ids_dict['train.json'], PBAE_subset_size)
    print("PBAE_subset: ", PBAE_subset)

    nonPBAE_subset = nonPBAE_lnp_ids_dict['train.json']
    print("nonPBAE_subset: ", nonPBAE_subset)

    PBAE_subset.extend(nonPBAE_subset)
    print("PBAE_subset: ", PBAE_subset)

    with open('processed_data_dirs/demo_in_house_lnp_data_overall_new_full_with_pbae_NPratios_updated_09222023_npratios_09252023gen_fig4bii/fold_V0/in_house_lnp/train_lnp_ids_PBAELNPtrain{}pct.json'.format(p), 'w') as fp:
        json.dump(PBAE_subset, fp)

In [ ]:
# save test and valid lnp_ids into separate json files

test_lnp_ids = all_lnp_ids_dict['test.json']
with open('processed_data_dirs/demo_in_house_lnp_data_overall_new_full_with_pbae_NPratios_updated_09222023_npratios_09252023gen_fig4bii/fold_V0/in_house_lnp/test_lnp_ids.json', 'w') as fp:
    json.dump(test_lnp_ids, fp)


valid_lnp_ids = all_lnp_ids_dict['valid.json']
with open('processed_data_dirs/demo_in_house_lnp_data_overall_new_full_with_pbae_NPratios_updated_09222023_npratios_09252023gen_fig4bii/fold_V0/in_house_lnp/valid_lnp_ids.json', 'w') as fp:
    json.dump(valid_lnp_ids, fp)